In [2]:
## 1. DAGSHUB & MLFLOW INITIALIZATION
!pip install dagshub mlflow lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import mlflow
import dagshub
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 96.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 95.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=bd4bf280-c6c8-449c-9765-97281046c52b&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c0f7029fad7edca33a302d1f95a6a9cb4869898b1a37676163ebfbbd4b4209e0




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [3]:
## Data Load and Split
DATA_BASE_PATH = '/kaggle/input/notebooks/elenejobava/walmart-eda-feature-engineering/'

train_fe = pd.read_parquet(DATA_BASE_PATH + 'train_features.parquet')
test_fe  = pd.read_parquet(DATA_BASE_PATH + 'test_features.parquet')

# Kaggle evaluation metric is Weighted Mean Absolute Error (WMAE)
# Holiday weeks have a weight of 5, normal weeks have a weight of 1
train_fe['weight'] = np.where(train_fe['IsHoliday'] == 1, 5, 1)

# Drop non-feature columns
features = [c for c in train_fe.columns if c not in ['Weekly_Sales', 'Date', 'Store_Dept', 'Type', 'weight']]
target = 'Weekly_Sales'

# Use the last 3 months of train data as our validation set.
split_date = train_fe['Date'].max() - pd.Timedelta(weeks=12)

X_train = train_fe[train_fe['Date'] <= split_date][features]
y_train = train_fe[train_fe['Date'] <= split_date][target]
w_train = train_fe[train_fe['Date'] <= split_date]['weight']

X_val = train_fe[train_fe['Date'] > split_date][features]
y_val = train_fe[train_fe['Date'] > split_date][target]
w_val = train_fe[train_fe['Date'] > split_date]['weight']

print(f"Training shape: {X_train.shape}, Validation shape: {X_val.shape}")

# Create LightGBM Dataset objects (highly memory efficient)
train_data = lgb.Dataset(X_train, label=y_train, weight=w_train)
val_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data)

Training shape: (386007, 78), Validation shape: (35563, 78)


In [5]:
## MODEL TRAINING & TRACKING
# Autologging handles saving the model, parameters, and feature importance automatically
mlflow.lightgbm.autolog()

# Set up the parent experiment per your professor's instructions
mlflow.set_experiment("LightGBM_Training")

# Define hyperparameter dictionary
params = {
    'objective': 'regression',
    'metric': 'mae',          # Mean Absolute Error
    'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
    'learning_rate': 0.05,    # Step size 
    'num_leaves': 63,         # Max leaves per tree (control complexity)
    'feature_fraction': 0.8,  # Randomly sample 80% of features per tree (prevents overfitting)
    'verbose': -1,            # Suppress internal LightGBM warnings
    'random_state': 42
}

with mlflow.start_run(run_name="LightGBM_Baseline"):
    
    # Train the model
    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'validation'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    # Generate validation predictions for final manual metric check
    val_preds = model.predict(X_val)
    
    # Calculate Custom Kaggle WMAE
    wmae = np.sum(w_val * np.abs(y_val - val_preds)) / np.sum(w_val)
    
    # Log the custom final metric to MLflow
    mlflow.log_metric("validation_WMAE", wmae)
    
    print(f"\n✓ Validation WMAE: {wmae:.2f}")

Training until validation scores don't improve for 50 rounds
[50]	train's l1: 1709.17	validation's l1: 1735.78
[100]	train's l1: 1034.38	validation's l1: 1039.5
[150]	train's l1: 955.619	validation's l1: 984.568
[200]	train's l1: 907.55	validation's l1: 956.625
[250]	train's l1: 877.453	validation's l1: 945.981
[300]	train's l1: 852.453	validation's l1: 937.933
[350]	train's l1: 828.575	validation's l1: 928.045
[400]	train's l1: 807.785	validation's l1: 921.66
[450]	train's l1: 789.626	validation's l1: 916.477
[500]	train's l1: 773.863	validation's l1: 911.737
[550]	train's l1: 759.334	validation's l1: 908.641
[600]	train's l1: 746.347	validation's l1: 905.549
[650]	train's l1: 733.875	validation's l1: 902.171
[700]	train's l1: 722.745	validation's l1: 899.319
[750]	train's l1: 712.344	validation's l1: 896.894
[800]	train's l1: 702.969	validation's l1: 895.35
[850]	train's l1: 693.774	validation's l1: 894.897
[900]	train's l1: 684.986	validation's l1: 892.904
[950]	train's l1: 676.476	

2026/07/12 12:28:36 WARNING mlflow.lightgbm: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/07/12 12:28:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



✓ Validation WMAE: 890.16
🏃 View run LightGBM_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/1/runs/0c19cb95e7914cd382adca511c5b402d
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/1


In [3]:
## RUN 4: OPTIMIZED LIGHTGBM & WAPE METRIC
mlflow.set_experiment("LightGBM_Training")

def calculate_wape(y_true, y_pred):
    """Calculates Weighted Absolute Percentage Error (Retail Standard)"""
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

# Relaxed parameters: Giving the model enough capacity to learn the massive dataset
params_optimized = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 127,             # [FIX] Increased to allow complex pattern learning
    'min_data_in_leaf': 20,        # [FIX] Relaxed restriction
    'lambda_l1': 0.01,             # [FIX] Lighter penalty
    'lambda_l2': 0.01,             # [FIX] Lighter penalty
    'feature_fraction': 0.8,
    'feature_pre_filter': False,   # [THE FIX] Forces LightGBM to ignore cached dataset filters
    'verbose': -1,
    'random_state': 42
}

print("Starting Optimized MLflow run...")

# Force MLflow to re-attach the autologger for this specific cell
mlflow.lightgbm.autolog()

with mlflow.start_run(run_name="LightGBM_Optimized"):
    
    # Train the restricted model
    model_opt = lgb.train(
        params_optimized,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'validation'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )
    
    # Generate predictions
    val_preds_opt = model_opt.predict(X_val)
    
    # Calculate Metrics
    wmae_opt = np.sum(w_val * np.abs(y_val - val_preds_opt)) / np.sum(w_val)
    wape_opt = calculate_wape(y_val, val_preds_opt)
    
    # Log custom metrics
    mlflow.log_metric("validation_WMAE", wmae_opt)
    mlflow.log_metric("validation_WAPE_percent", wape_opt)
    
    print(f"Validation WMAE: {wmae_opt:.2f}")
    print(f"Human-Readable Error (WAPE): {wape_opt:.2f}%")

Starting Optimized MLflow run...
Training until validation scores don't improve for 50 rounds
[50]	train's l1: 1662.27	validation's l1: 1688.42
[100]	train's l1: 956.436	validation's l1: 994.217
[150]	train's l1: 868.502	validation's l1: 944.987
[200]	train's l1: 815.293	validation's l1: 927.61
[250]	train's l1: 780.564	validation's l1: 918.15
[300]	train's l1: 749.379	validation's l1: 908.742
[350]	train's l1: 724.672	validation's l1: 904.365
[400]	train's l1: 703.233	validation's l1: 900.411
[450]	train's l1: 684.039	validation's l1: 897.561
[500]	train's l1: 666.62	validation's l1: 895.357
[550]	train's l1: 651.102	validation's l1: 893.409
[600]	train's l1: 637.113	validation's l1: 891.501
[650]	train's l1: 624.806	validation's l1: 891.241
[700]	train's l1: 613.037	validation's l1: 889.842
[750]	train's l1: 601.546	validation's l1: 887.905
[800]	train's l1: 591.412	validation's l1: 887.787
Early stopping, best iteration is:
[764]	train's l1: 598.567	validation's l1: 887.31


2026/07/04 09:04:05 WARNING mlflow.lightgbm: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/07/04 09:04:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Validation WMAE: 887.31
Human-Readable Error (WAPE): 5.67%
🏃 View run LightGBM_Optimized at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/0/runs/5a85886b9a0f41a69d85369ff0aba13c
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/0


In [7]:
## 5. SKLEARN PIPELINE & MODEL REGISTRY
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
import mlflow.sklearn


# 1. Create the Custom Data Engineer Class
class WalmartFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Initialize empty variables to memorize training statistics
        self.dept_avg_sales_ = None
        self.store_avg_sales_ = None
        self.store_dept_avg_sales_ = None
        
    def fit(self, X, y=None):
        # Temporarily attach the target variable to calculate means
        X_temp = X.copy()
        X_temp['Weekly_Sales'] = y
        
        # Memorize the statistics
        self.dept_avg_sales_ = X_temp.groupby('Dept')['Weekly_Sales'].mean()
        self.store_avg_sales_ = X_temp.groupby('Store')['Weekly_Sales'].mean()
        self.store_dept_avg_sales_ = X_temp.groupby(['Store', 'Dept'])['Weekly_Sales'].mean()
        
        return self
        
    def transform(self, X):
        df = X.copy()
        
        # A. Apply Stateless Features (Math that doesn't need historical context)
        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['Week_sin'] = np.sin(2 * np.pi * df['Week'] / 52)
        df['IsHoliday'] = df['IsHoliday'].astype(int)
        
        # B. Apply Stateful Features (Mapping the memorized training data)
        df['dept_avg_sales'] = df['Dept'].map(self.dept_avg_sales_)
        df['store_avg_sales'] = df['Store'].map(self.store_avg_sales_)
        df['store_dept_avg_sales'] = df.set_index(['Store', 'Dept']).index.map(self.store_dept_avg_sales_)
        
        # Handle 'Cold Start' NaNs for departments/stores that didn't exist in the training set
        df['dept_avg_sales'] = df['dept_avg_sales'].fillna(0)
        df['store_avg_sales'] = df['store_avg_sales'].fillna(0)
        df['store_dept_avg_sales'] = df['store_dept_avg_sales'].fillna(df['dept_avg_sales'])
        
        # Drop columns LightGBM cannot read
        cols_to_drop = [c for c in ['Date', 'Type'] if c in df.columns]
        df = df.drop(columns=cols_to_drop)
        
        return df

# 2. Define the Optimized LightGBM parameters (from my previous run)
pipeline_params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 127,             
    'min_data_in_leaf': 20,        
    'lambda_l1': 0.01,             
    'lambda_l2': 0.01,             
    'feature_fraction': 0.8,
    'random_state': 42,
    'n_estimators': 764 # From my early stopping best iteration
}

# 3. Stitch the Feature Engineer and the Model together
full_pipeline = Pipeline([
    ('feature_engineering', WalmartFeatureEngineer()),
    ('model', LGBMRegressor(**pipeline_params))
])

# 6. TRAIN & REGISTER IN MLFLOW

# We need the RAW, un-engineered training data to prove the pipeline works
# Load raw train data and merge with raw features/stores
raw_train = pd.read_csv('/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip', parse_dates=['Date'])
raw_features = pd.read_csv('/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip', parse_dates=['Date'])
raw_stores = pd.read_csv('/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv')

# Basic merge so all columns are available
raw_data = raw_train.merge(raw_features, on=['Store', 'Date'], how='left', suffixes=('', '_feat')).merge(raw_stores, on='Store', how='left')
raw_X = raw_data.drop(columns=['Weekly_Sales', 'IsHoliday_feat'])
raw_y = raw_data['Weekly_Sales']

with mlflow.start_run(run_name="Final_LightGBM_Pipeline"):
    
    # Train the entire pipeline at once
    full_pipeline.fit(raw_X, raw_y)
    
    # Log and Register the model simultaneously
    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        artifact_path="model",
        registered_model_name="Walmart_LightGBM_Production_Pipeline",
        serialization_format="cloudpickle"
    )
    

2026/07/12 12:45:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 12:45:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 12:45:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Walmart_LightGBM_Production_Pipeline'.
2026/07/12 12:45:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_LightGBM_Production_Pipeline, version 1
Created version '1' of model 'Walmart_LightGBM_Production_Pipeline'.


🏃 View run Final_LightGBM_Pipeline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/1/runs/69c20899b864457fb8b9e596e92d20f5
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/1
